In [1]:
# TensorFlow / Keras imports
import tensorflow as tf

# Model
from tensorflow.keras.models import Model

# Input layer
from tensorflow.keras.layers import Input

# Convolution layers
from tensorflow.keras.layers import Conv2D

# Normalization
from tensorflow.keras.layers import BatchNormalization

# Activation
from tensorflow.keras.layers import LeakyReLU

# Padding
from tensorflow.keras.layers import ZeroPadding2D

# Upsampling for feature maps
from tensorflow.keras.layers import UpSampling2D

# Feature map merge
from tensorflow.keras.layers import add
from tensorflow.keras.layers import concatenate

In [2]:
def _conv_block(inp, convs, skip=True):  
    # Function banaya jo convolution block create karega
    # inp = input tensor (previous layer ka output)
    # convs = convolution layers ki list (configuration)
    # skip = kya skip connection use karna hai ya nahi

    x = inp  
    # x me input store kar diya, ab isi par layers apply hongi

    count = 0  
    # count variable loop me layer number track karega

    for conv in convs:  
        # convs list ke har convolution configuration par loop chalega

        if count == (len(convs) - 2) and skip:  
            # agar current layer second-last layer hai aur skip True hai
            skip_connection = x  
            # to current output ko skip connection ke liye store kar lo

        count += 1  
        # har iteration me count 1 se badhega

        if conv['stride'] > 1:  
            # agar stride 1 se bada hai (downsampling ho raha hai)
            x = ZeroPadding2D(((1,0),(1,0)))(x)  
            # to padding add karo (Darknet style padding)

        x = Conv2D(  
            conv['filter'],  
            # kitne filters use honge

            conv['kernel'],  
            # kernel size (jaise 3x3, 1x1)

            strides=conv['stride'],  
            # stride value

            padding='valid' if conv['stride'] > 1 else 'same',  
            # agar stride >1 hai to valid padding
            # warna same padding

            name='conv_' + str(conv['layer_idx']),  
            # layer ka unique naam banaya

            use_bias=False if conv['bnorm'] else True  
            # agar batch normalization use ho raha hai to bias ki zarurat nahi
        )(x)  
        # convolution layer apply karke output x me store

        if conv['bnorm']:  
            # agar batch normalization enable hai
            x = BatchNormalization(  
                epsilon=0.001,  
                # small value for numerical stability
                name='bnorm_' + str(conv['layer_idx'])  
                # batchnorm layer ka naam
            )(x)  
            # batch normalization apply

        if conv['leaky']:  
            # agar leaky relu activation enable hai
            x = LeakyReLU(  
                alpha=0.1,  
                # negative slope value
                name='leaky_' + str(conv['layer_idx'])  
                # activation layer ka naam
            )(x)  
            # activation apply

    return add([skip_connection, x]) if skip else x  
    # agar skip=True hai to skip connection add karo
    # matlab previous stored feature + current feature
    # warna sirf x return karo

### Concept of `_conv_block`

Is code ka purpose ek **CNN block banana** hai jo image se useful features extract karta hai aur deep neural network ko stable train karne me help karta hai.

Sabse pehle function **input tensor (`inp`)** leta hai jo previous layer ka output hota hai. Phir `convs` list ke according loop chalakar **multiple convolution layers** apply ki jati hain. Convolution layers ka kaam image se **edges, shapes aur patterns** jaise important features nikalna hota hai.

Har convolution ke baad **Batch Normalization** lagayi jati hai taaki training stable aur fast ho. Uske baad **LeakyReLU activation** use hota hai jo model ko complex aur non-linear patterns samajhne me help karta hai.

Agar convolution ka **stride 1 se bada ho** to **Zero Padding** apply ki jati hai taaki feature map ka structure properly maintain rahe.

End me agar `skip=True` ho to **skip connection** use hota hai jisme previous feature map aur current feature map ko **add** kar diya jata hai. Isse deep networks me **information loss aur vanishing gradient problem** kam hoti hai aur model better learn karta hai.

Simple words me, ye function **Conv → BatchNorm → Activation → Skip connection** ka ek reusable CNN block banata hai jo deep learning models (jaise YOLO / Darknet) me feature extraction ke liye use hota hai.

### _conv_block in YOLO

Is code ka purpose YOLO model ke liye ek **convolution block** banana hai jo image se important features extract karta hai.

Sabse pehle function input tensor (`inp`) leta hai jo previous layer ka output hota hai. Phir `convs` list ke according multiple **Conv2D layers** apply hoti hain jo image se edges, shapes aur object patterns jaise features detect karti hain.

Har convolution ke baad **Batch Normalization** use hota hai taaki training stable aur fast ho, aur **LeakyReLU activation** model ko complex patterns samajhne me help karta hai.

Agar stride 1 se bada ho to **Zero Padding** apply ki jati hai taaki feature map ka structure maintain rahe.

End me **skip connection** use hota hai jisme previous feature map aur current feature map ko add kiya jata hai. Isse deep network me information loss kam hota hai aur model better learn karta hai.

Is tarah ka convolution block YOLO architecture me **feature extraction backbone** banane ke liye use hota hai.

## _conv_block Pipeline (YOLO)

**Purpose:** Image se useful features extract karna aur deep network ko stable banana.

### Flow

Input Feature Map  
↓  
Check stride  
→ agar stride > 1 ho to **ZeroPadding**  
↓  
**Conv2D**  
(feature extraction: edges, shapes, patterns)  
↓  
**Batch Normalization**  
(training stable aur fast)  
↓  
**LeakyReLU Activation**  
(non-linear patterns learn karne ke liye)  
↓  
Repeat for all convolution configs  
↓  
**Skip Connection**  
(previous feature + current feature)  
↓  
Output Feature Map

---

### Short Idea

**Input → Conv → BatchNorm → LeakyReLU → Repeat → Skip Connection → Output**

Ye block YOLO backbone me use hota hai taaki **deep layers better features learn kare aur information loss na ho.**

In [3]:
def make_yolov3_model():
    input_image = Input(shape=(None, None, 3))

    # Layer 0 => 4
    x = _conv_block(input_image, [
        {'filter': 32, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 0},
        {'filter': 64, 'kernel': 3, 'stride': 2, 'bnorm': True, 'leaky': True, 'layer_idx': 1},
        {'filter': 32, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 2},
        {'filter': 64, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 3}
    ])

    # Layer 5 => 8
    x = _conv_block(x, [
        {'filter': 128, 'kernel': 3, 'stride': 2, 'bnorm': True, 'leaky': True, 'layer_idx': 5},
        {'filter': 64, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 6},
        {'filter': 128, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 7}
    ])

    # Layer 9 => 11
    x = _conv_block(x, [
        {'filter': 64, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 9},
        {'filter': 128, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 10}
    ])

    # Layer 12 => 15
    x = _conv_block(x, [
        {'filter': 256, 'kernel': 3, 'stride': 2, 'bnorm': True, 'leaky': True, 'layer_idx': 12},
        {'filter': 128, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 13},
        {'filter': 256, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 14}
    ])

    # Layer 16 => 36
    for i in range(7):
        x = _conv_block(x, [
            {'filter': 128, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 16+i*3},
            {'filter': 256, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 17+i*3}
        ])

    skip_36 = x

    # Layer 37 => 40
    x = _conv_block(x, [
        {'filter': 512, 'kernel': 3, 'stride': 2, 'bnorm': True, 'leaky': True, 'layer_idx': 37},
        {'filter': 256, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 38},
        {'filter': 512, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 39}
    ])

    # Layer 41 => 61
    for i in range(7):
        x = _conv_block(x, [
            {'filter': 256, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 41+i*3},
            {'filter': 512, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 42+i*3}
        ])

    skip_61 = x

    # Layer 62 => 65
    x = _conv_block(x, [
        {'filter': 1024, 'kernel': 3, 'stride': 2, 'bnorm': True, 'leaky': True, 'layer_idx': 62},
        {'filter': 512, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 63},
        {'filter': 1024, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 64}
    ])

    # Layer 66 => 74
    for i in range(3):
        x = _conv_block(x, [
            {'filter': 512, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 66+i*3},
            {'filter': 1024, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 67+i*3}
        ])

    # Layer 75 => 79
    x = _conv_block(x, [
        {'filter': 512, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 75},
        {'filter': 1024, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 76},
        {'filter': 512, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 77},
        {'filter': 1024, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 78},
        {'filter': 512, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 79}
    ], skip=False)

    # Layer 80 => 82
    yolo_82 = _conv_block(x, [
        {'filter': 1024, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 80},
        {'filter': 255, 'kernel': 1, 'stride': 1, 'bnorm': False, 'leaky': False, 'layer_idx': 81}
    ], skip=False)

    # Layer 83 => 86
    x = _conv_block(x, [
        {'filter': 256, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 84}
    ], skip=False)

    x = UpSampling2D(2)(x)
    x = concatenate([x, skip_61])

    # Layer 87 => 91
    x = _conv_block(x, [
        {'filter': 256, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 87},
        {'filter': 512, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 88},
        {'filter': 256, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 89},
        {'filter': 512, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 90},
        {'filter': 256, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 91}
    ], skip=False)

    # Layer 92 => 94
    yolo_94 = _conv_block(x, [
        {'filter': 512, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 92},
        {'filter': 255, 'kernel': 1, 'stride': 1, 'bnorm': False, 'leaky': False, 'layer_idx': 93}
    ], skip=False)

    # Layer 95 => 98
    x = _conv_block(x, [
        {'filter': 128, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 96}
    ], skip=False)

    x = UpSampling2D(2)(x)
    x = concatenate([x, skip_36])

    # Layer 99 => 106
    yolo_106 = _conv_block(x, [
        {'filter': 128, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 99},
        {'filter': 256, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 100},
        {'filter': 128, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 101},
        {'filter': 256, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 102},
        {'filter': 128, 'kernel': 1, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 103},
        {'filter': 256, 'kernel': 3, 'stride': 1, 'bnorm': True, 'leaky': True, 'layer_idx': 104},
        {'filter': 255, 'kernel': 1, 'stride': 1, 'bnorm': False, 'leaky': False, 'layer_idx': 105}
    ], skip=False)

    model = Model(input_image, [yolo_82, yolo_94, yolo_106])
    return model

## YOLOv3 Model Architecture (make_yolov3_model)

Is function ka purpose **YOLOv3 object detection model build karna** hai.

Sabse pehle model **input image** leta hai jiska shape `(None, None, 3)` hota hai, matlab image ka size flexible ho sakta hai aur usme 3 color channels (RGB) hote hain.

Phir multiple **convolution blocks (_conv_block)** use kiye jate hain jo image se features extract karte hain. Ye blocks YOLO backbone (Darknet-53) ka part hote hain aur image se gradually **low-level se high-level features** learn karte hain.

Architecture me kuch jagah **skip connections (skip_36, skip_61)** store kiye jate hain taaki baad me unhe concatenate karke **multi-scale feature fusion** kiya ja sake.

Phir **UpSampling2D** use hota hai jisse feature maps ko bada kiya jata hai aur unhe previous layers ke features ke saath **concatenate** kiya jata hai. Isse model different object sizes detect kar pata hai.

YOLOv3 me **3 detection heads** hote hain:

- **yolo_82** → large objects detect karta hai  
- **yolo_94** → medium objects detect karta hai  
- **yolo_106** → small objects detect karta hai  

Finally model **three output feature maps** return karta hai jo bounding boxes, objectness score aur class probabilities predict karte hain.

In [4]:
model = make_yolov3_model()
model.summary()

d:\DP Larning\.venv\lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0 (Conv2D)     │ (None, None,      │        864 │ input_layer[0][0] │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bnorm_0             │ (None, None,      │        128 │ conv_0[0][0]      │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_0 (LeakyReLU) │ (None, None,      │          0 │ bnorm_0[0][0]     │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, None,      │          0 │ leaky_0[0][0]     │
│ (ZeroPadding2D)     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv2D)     │ (None, None,      │     18,432 │ zero_padding2d[0… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bnorm_1             │ (None, None,      │        256 │ conv_1[0][0]      │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_1 (LeakyReLU) │ (None, None,      │          0 │ bnorm_1[0][0]     │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv2D)     │ (None, None,      │      2,048 │ leaky_1[0][0]     │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bnorm_2             │ (None, None,      │        128 │ conv_2[0][0]      │
│ (BatchNormalizatio… │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_2 (LeakyReLU) │ (None, None,      │          0 │ bnorm_2[0][0]     │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv2D)     │ (None, None,      │     18,432 │ leaky_2[0][0]     │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bnorm_3             │ (None, None,      │        256 │ conv_3[0][0]      │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_3 (LeakyReLU) │ (None, None,      │          0 │ bnorm_3[0][0]     │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None,      │          0 │ leaky_1[0][0],    │
│                     │ None, 64)         │            │ leaky_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, None,      │          0 │ add[0][0]         │
│ (ZeroPadding2D)     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_5 (Conv2D)     │ (None, None,      │     73,728 │ zero_padding2d_1

 Total params: 62,001,757 (236.52 MB)

 Trainable params: 61,949,149 (236.32 MB)

 Non-trainable params: 52,608 (205.50 KB)

# YOLOv3 Model Pipeline

## 1️⃣ Input Image
Model sabse pehle **input image** leta hai.

Input Shape:
Image → (Height, Width, 3)

3 ka matlab:
RGB channels

---

## 2️⃣ Feature Extraction (Backbone - Darknet53)

Image ko multiple **convolution blocks** se pass karaya jata hai.

Conv Block me hota hai:

Conv2D  
↓  
BatchNormalization  
↓  
LeakyReLU  

Ye layers image se **important features** nikalti hain:

edges  
textures  
shapes  
object parts  

Deep layers me **high-level features** learn hote hain.

---

## 3️⃣ Residual / Skip Connections

Kuch layers ke output store kiye jate hain:

skip_36  
skip_61  

Baad me inhe use karke **feature maps combine** kiye jate hain.

Purpose:

information loss kam karna  
deep network ko stable banana

---

## 4️⃣ Multi-Scale Feature Learning

YOLOv3 different object sizes detect karta hai.

Isliye model:

Upsampling  
↓  
Concatenation  
↓  
Feature fusion

use karta hai.

Upsampling → feature map ko bada karta hai  
Concatenate → different layers ke features combine karta hai

---

## 5️⃣ Detection Heads

YOLOv3 me **3 detection layers** hoti hain.

### Detection Layer 1
yolo_82  
Large objects detect karta hai

---

### Detection Layer 2
yolo_94  
Medium objects detect karta hai

---

### Detection Layer 3
yolo_106  
Small objects detect karta hai

---

## 6️⃣ Final Output

Model 3 feature maps return karta hai:

yolo_82  
yolo_94  
yolo_106  

Har detection layer predict karti hai:

Bounding box  
Object confidence  
Class probability

---

# Final Flow

Input Image  
↓  
Darknet53 Backbone  
↓  
Residual Connections  
↓  
Feature Fusion (Upsampling + Concatenate)  
↓  
3 Detection Heads  
↓  
Bounding Box Predictions

## YOLOv3 Model Pipeline (Simple)

**1. Input Image**  
Model input image leta hai `(H, W, 3)`.

↓  

**2. Feature Extraction (Backbone)**  
Image ko multiple **convolution blocks** se pass kiya jata hai.  
Yahan Conv2D + BatchNorm + LeakyReLU use hote hain taaki image se features (edges, shapes, patterns) nikale ja sake.

↓  

**3. Skip Connections**  
Kuch intermediate feature maps store kiye jate hain (`skip_36`, `skip_61`).  
Baad me inhe use karke deep layers ke features ko combine kiya jata hai.

↓  

**4. Upsampling + Concatenation**  
Feature maps ko **UpSampling** se bada kiya jata hai aur previous features ke saath **concatenate** kiya jata hai.  
Isse model different object sizes detect kar pata hai.

↓  

**5. Detection Layers**  

- `yolo_82` → large objects  
- `yolo_94` → medium objects  
- `yolo_106` → small objects  

↓  

**6. Output**  
Model 3 outputs deta hai jo predict karte hain:

- Bounding box  
- Object confidence  
- Class probability